# 🤖 Tunisian ASR — Model Fine-Tuning (`04_model_training`)

Fine-tune a pretrained ASR model on the prepared Tunisian Arabic dataset.

**Two supported model families:**
- **Whisper** (OpenAI) — encoder-decoder, sequence-to-sequence
- **XLSR-Wav2Vec2** (Meta / HuggingFace) — CTC-based encoder-only

Fill in the `TODO` sections with your chosen model name and tweak hyperparameters.
Everything else is wired and ready to run.

### Road-map
| § | Section |
|---|---------|
| 1 | Environment & model selection |
| 2 | Load final dataset |
| 3 | Processor / feature extractor + tokenizer |
| 4 | Data collator |
| 5 | Compute metrics |
| 6 | Training arguments |
| 7 | Trainer |
| 8 | Training loop |
| 9 | Evaluation |
| 10 | Save & push |
| 11 | Plot training metrics |

## 1 · Environment & Model Selection

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, warnings, os
warnings.filterwarnings("ignore")
from pathlib import Path

PROJECT_ROOT = str(Path.cwd().parent.resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
from datasets import load_from_disk, Audio

from src.utils import load_config, print_dict, print_section

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}  "
      f"({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'})")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# TODO: choose your model family and checkpoint
# ─────────────────────────────────────────────────────────────────────────
# Option A — Whisper (recommended for Tunisian Arabic)
MODEL_FAMILY   = "whisper"                         # "whisper" or "wav2vec2"
MODEL_NAME     = "openai/whisper-small"            # any whisper variant
LANGUAGE       = "arabic"
TASK           = "transcribe"

# Option B — XLSR-Wav2Vec2 (CTC)
# MODEL_FAMILY = "wav2vec2"
# MODEL_NAME   = "facebook/wav2vec2-large-xlsr-53"

# ── Output directory ───────────────────────────────────────────────────────
paths         = load_config(f"{PROJECT_ROOT}/configs/paths.yaml")
DATASET_PATH  = Path(PROJECT_ROOT) / paths["processed"]["final"]
OUTPUT_DIR    = Path(PROJECT_ROOT) / "outputs" / "models" / Path(MODEL_NAME).name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model family  : {MODEL_FAMILY}")
print(f"Checkpoint    : {MODEL_NAME}")
print(f"Dataset       : {DATASET_PATH}")
print(f"Output dir    : {OUTPUT_DIR}")

## 2 · Load Final Dataset

In [ ]:
dataset = load_from_disk(str(DATASET_PATH))
print(dataset)

# Resample to 16 kHz (already done in Phase 2 — this is a safety guard)
TARGET_SR = 16_000
dataset = dataset.cast_column("audio", Audio(sampling_rate=TARGET_SR))

print(f"\nTrain : {len(dataset['train']):,} segments")
print(f"Test  : {len(dataset['test']):,}  segments")
print(f"\nFeatures: {dataset['train'].features}")

In [ ]:
# Inspect one sample
sample = dataset["train"][0]
print(f"audio_id   : {sample['audio_id']}")
print(f"transcript : {sample['transcript']}")
print(f"audio.sr   : {sample['audio']['sampling_rate']}")
print(f"audio.shape: {np.array(sample['audio']['array']).shape}")

## 3 · Processor / Feature Extractor + Tokenizer

In [ ]:
if MODEL_FAMILY == "whisper":
    from transformers import WhisperProcessor, WhisperForConditionalGeneration

    processor = WhisperProcessor.from_pretrained(MODEL_NAME,
                                                  language=LANGUAGE,
                                                  task=TASK)
    feature_extractor = processor.feature_extractor
    tokenizer         = processor.tokenizer

    model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
    # Force Arabic generation (prevents fallback to English)
    model.generation_config.language = LANGUAGE
    model.generation_config.task     = TASK
    model.generation_config.forced_decoder_ids = None

elif MODEL_FAMILY == "wav2vec2":
    from transformers import (Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor,
                               Wav2Vec2Processor, Wav2Vec2ForCTC)

    # ── Build vocabulary from the training transcripts ─────────────────────
    # TODO: replace with the actual character set from your final dataset
    all_chars = set("".join(dataset["train"]["transcript"]))
    vocab = {c: i for i, c in enumerate(sorted(all_chars))}
    vocab.update({"[UNK]": len(vocab), "[PAD]": len(vocab) + 1, "|": len(vocab) + 2})
    vocab_path = OUTPUT_DIR / "vocab.json"
    import json
    with open(vocab_path, "w", encoding="utf-8") as f:
        json.dump(vocab, f, ensure_ascii=False)

    tokenizer = Wav2Vec2CTCTokenizer(
        str(vocab_path), unk_token="[UNK]", pad_token="[PAD]",
        word_delimiter_token="|",
    )
    feature_extractor = Wav2Vec2FeatureExtractor(
        feature_size=1, sampling_rate=TARGET_SR, padding_value=0.0,
        do_normalize=True, return_attention_mask=True,
    )
    processor = Wav2Vec2Processor(
        feature_extractor=feature_extractor, tokenizer=tokenizer
    )
    model = Wav2Vec2ForCTC.from_pretrained(
        MODEL_NAME,
        ctc_loss_reduction="mean",
        pad_token_id=processor.tokenizer.pad_token_id,
        vocab_size=len(processor.tokenizer),
        ignore_mismatched_sizes=True,
    )
    # Freeze feature encoder for first N steps (common practice)
    model.freeze_feature_encoder()

print(f"Model parameters : {model.num_parameters():,}")
print(f"Trainable        : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 4 · Preprocessing Function & Data Collator

In [ ]:
def preprocess_batch(batch):
    """Map function: audio array → model inputs + labels."""
    audio_arrays = [np.array(a["array"], dtype=np.float32)
                    for a in batch["audio"]]

    if MODEL_FAMILY == "whisper":
        # Feature extractor converts raw audio → log-mel spectrogram
        inputs = feature_extractor(audio_arrays, sampling_rate=TARGET_SR,
                                   return_tensors="pt").input_features
        # Tokenizer converts transcript → label ids
        labels = tokenizer(batch["transcript"], return_tensors="pt").input_ids
        batch["input_features"] = inputs
        batch["labels"]         = labels

    elif MODEL_FAMILY == "wav2vec2":
        inputs = processor(audio_arrays, sampling_rate=TARGET_SR,
                           return_tensors="pt", padding=True)
        batch["input_values"]     = inputs.input_values[0]
        batch["attention_mask"]   = inputs.attention_mask[0]
        with processor.as_target_processor():
            labels = processor(batch["transcript"],
                               return_tensors="pt").input_ids
        batch["labels"] = labels[0]

    return batch

# Apply preprocessing (use num_proc=4 if you have multiple CPU cores)
dataset = dataset.map(
    preprocess_batch,
    batched=True,
    batch_size=8,
    remove_columns=["audio"],   # raw audio bytes no longer needed
    desc="Preprocessing",
)
print("Preprocessing done.")
print("Columns:", dataset["train"].column_names)

In [ ]:
import dataclasses
from typing import Any, Dict, List, Union

if MODEL_FAMILY == "whisper":
    from transformers import DataCollatorSpeechSeq2SeqWithPadding
    data_collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
    )

elif MODEL_FAMILY == "wav2vec2":
    from dataclasses import dataclass

    @dataclass
    class DataCollatorCTCWithPadding:
        processor: Any
        padding: Union[bool, str] = True

        def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
            import torch
            input_features = [{"input_values": f["input_values"]} for f in features]
            label_features = [{"input_ids": f["labels"]}           for f in features]

            batch = self.processor.pad(input_features, padding=self.padding,
                                       return_tensors="pt")
            with self.processor.as_target_processor():
                labels_batch = self.processor.pad(label_features,
                                                   padding=self.padding,
                                                   return_tensors="pt")

            labels = labels_batch["input_ids"].masked_fill(
                labels_batch.attention_mask.ne(1), -100
            )
            batch["labels"] = labels
            return batch

    data_collator = DataCollatorCTCWithPadding(processor=processor)

print(f"Data collator: {type(data_collator).__name__}")

## 5 · Evaluation Metrics

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    """Compute WER and CER from model predictions."""
    pred_ids      = pred.predictions
    label_ids     = pred.label_ids

    # Replace -100 (padding) so the tokenizer can decode
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    if MODEL_FAMILY == "whisper":
        pred_str  = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    else:
        pred_ids  = np.argmax(pred_ids, axis=-1)
        pred_str  = processor.batch_decode(pred_ids)
        label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": round(wer, 4), "cer": round(cer, 4)}

print("Metrics: WER + CER")

## 6 · Training Arguments

In [ ]:
from transformers import Seq2SeqTrainingArguments, TrainingArguments

# ── TODO: tune these hyperparameters ────────────────────────────────────────
# Recommended starting points for Whisper-small on ~80 h of data:
#   batch_size=16, learning_rate=1e-5, warmup_steps=500, max_steps=4000
#   gradient_accumulation_steps=2 if GPU VRAM < 16 GB

if MODEL_FAMILY == "whisper":
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(OUTPUT_DIR),
        per_device_train_batch_size=16,           # TODO: adjust to GPU VRAM
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=1e-5,
        warmup_steps=500,
        max_steps=4000,
        gradient_checkpointing=True,
        fp16=torch.cuda.is_available(),           # use float16 on GPU
        evaluation_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        logging_steps=25,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=225,
        report_to=["tensorboard"],
        push_to_hub=False,
    )

elif MODEL_FAMILY == "wav2vec2":
    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR),
        group_by_length=True,
        per_device_train_batch_size=16,           # TODO: adjust to GPU VRAM
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,
        evaluation_strategy="steps",
        num_train_epochs=30,
        fp16=torch.cuda.is_available(),
        save_steps=400,
        eval_steps=400,
        logging_steps=50,
        learning_rate=3e-4,
        warmup_steps=500,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        report_to=["tensorboard"],
        push_to_hub=False,
    )

print(training_args)

## 7 · Trainer

In [ ]:
from transformers import Seq2SeqTrainer, Trainer

TrainerClass = Seq2SeqTrainer if MODEL_FAMILY == "whisper" else Trainer

trainer = TrainerClass(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=feature_extractor,   # for padding collation
)

print(f"Trainer class   : {TrainerClass.__name__}")
print(f"Train samples   : {len(dataset['train']):,}")
print(f"Eval  samples   : {len(dataset['test']):,}")
print(f"Estimated steps : {training_args.max_steps if hasattr(training_args, 'max_steps') else 'N/A'}")

## 8 · Training Loop

In [ ]:
# ── Optional: resume from a checkpoint ────────────────────────────────────
# resume_from = OUTPUT_DIR / "checkpoint-2000"
# trainer.train(resume_from_checkpoint=str(resume_from))

trainer.train()

## 9 · Evaluation

In [ ]:
eval_results = trainer.evaluate()
print_dict(eval_results, title="Evaluation — test split")

# Decode a few predictions for qualitative inspection
predictions = trainer.predict(dataset["test"].select(range(20)))
if MODEL_FAMILY == "whisper":
    pred_ids  = predictions.predictions
    label_ids = predictions.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
else:
    pred_ids  = np.argmax(predictions.predictions, axis=-1)
    pred_str  = processor.batch_decode(pred_ids)
    label_ids = predictions.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

print("\nSample predictions:")
print(f"{'Reference':<60}  {'Prediction':<60}")
print("-" * 124)
for ref, pred in zip(label_str[:10], pred_str[:10]):
    print(f"  {ref[:58]:<60}  {pred[:58]:<60}")

## 10 · Save & Push

In [ ]:
# Save model + processor
trainer.save_model(str(OUTPUT_DIR / "final"))
processor.save_pretrained(str(OUTPUT_DIR / "final"))
print(f"✓ Model saved → {OUTPUT_DIR / 'final'}")

# TODO: uncomment to push to the HuggingFace Hub
# trainer.push_to_hub(commit_message="Fine-tuned on Tunisian Arabic")

## 11 · Plot Training Metrics

In [ ]:
import matplotlib.pyplot as plt

# Extract logged history from trainer state
log_history = trainer.state.log_history

# Separate train and eval logs
train_logs = [l for l in log_history if "loss"      in l and "eval_loss" not in l]
eval_logs  = [l for l in log_history if "eval_loss" in l]

if train_logs and eval_logs:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Loss curves
    axes[0].plot([l["step"] for l in train_logs],
                 [l["loss"] for l in train_logs], label="train")
    axes[0].plot([l["step"] for l in eval_logs],
                 [l["eval_loss"] for l in eval_logs], label="eval")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Step")
    axes[0].legend()

    # WER
    if any("eval_wer" in l for l in eval_logs):
        axes[1].plot([l["step"] for l in eval_logs if "eval_wer" in l],
                     [l["eval_wer"] for l in eval_logs if "eval_wer" in l],
                     color="#E74C3C")
        axes[1].set_title("WER (eval)"); axes[1].set_xlabel("Step")

    # CER
    if any("eval_cer" in l for l in eval_logs):
        axes[2].plot([l["step"] for l in eval_logs if "eval_cer" in l],
                     [l["eval_cer"] for l in eval_logs if "eval_cer" in l],
                     color="#2ECC71")
        axes[2].set_title("CER (eval)"); axes[2].set_xlabel("Step")

    plt.tight_layout()
    plt.savefig(f"{PROJECT_ROOT}/outputs/figures/training_metrics.png",
                bbox_inches="tight")
    plt.show()
    print("✓ Saved training_metrics.png")
else:
    print("No training logs found — run training first.")

In [ ]:
# Print best checkpoint info
best_ckpt = trainer.state.best_model_checkpoint
best_wer  = trainer.state.best_metric
print(f"Best checkpoint : {best_ckpt}")
print(f"Best WER        : {best_wer:.4f}" if best_wer else "Best WER: N/A")